# AVDD: train the face and voice detectors on Google Colab

Run the cells **top to bottom** (Runtime → Run all works too). Total time on the free **T4 GPU** is about **1.5–2.5 hours**, mostly downloading and decoding video.

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU*** → Save.
2. Keep this tab open: free Colab disconnects after about 90 min without activity in the tab.
3. Optional: add a Hugging Face token as a Colab secret named `HF_TOKEN` (key icon on the left) for faster downloads.

**What you get at the end:** `visual_head.npz` and `audio_head.npz` (the trained models, a few KB each), measured results, graphs, and an updated README. The last section pushes them to GitHub.

Progress is saved to your Google Drive (`MyDrive/avdd_cache`). If Colab disconnects, reconnect and **run all cells again**: finished parts are skipped.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive (keeps progress across disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['CACHE'] = '/content/drive/MyDrive/avdd_cache'   # embeddings: about 300 MB, survives disconnects
os.environ['DATA'] = '/content/avdd_data'                  # datasets: about 10 GB, local disk (fast)
os.environ['E2E'] = '/content/avdd_e2e'                    # held-out test files
os.makedirs(os.environ['CACHE'], exist_ok=True)

## 3. Get the code

In [ ]:
%cd /content
!test -d Multimodel_video_audio_deepfake || git clone -b v2-update https://github.com/Gutsyt/Multimodel_video_audio_deepfake.git
%cd /content/Multimodel_video_audio_deepfake
!git pull -q && git log --oneline -1

## 4. Install (about 2 min). PyTorch with CUDA is already in Colab; this adds the project and training tools.

In [ ]:
!which ffmpeg ffprobe || apt-get -qq install -y ffmpeg
!pip install -q -e ".[video,hf,train]"
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set')
except Exception:
    print('No HF_TOKEN secret: downloads still work, just slower')

## 5. Download the training data (about 10 GB, 10–30 min)
FaceForensics++ (1,000 real + 1,020 fake videos), Celeb-DF/FakeAVCeleb frames, WildDeepfake, ASVspoof 2019 + 5, In-the-Wild. Re-running skips finished files.

In [ ]:
!python scripts/download_data.py $DATA 2>&1 | grep --line-buffered -v Warning

## 6. Extract embeddings (the long step)
The big pretrained networks stay **frozen**. Every face crop or 4-second speech window becomes a 1024-number fingerprint, saved to Drive.
Lines look like `vis: 1088 embedded (25.3/s)`. A line `saved vis_ff 4021` means that source is done.

In [ ]:
!python scripts/train_heads.py embed-visual --data $DATA --cache $CACHE --device cuda 2>&1 | grep --line-buffered -E 'saved|embedded'

In [ ]:
!python scripts/train_heads.py embed-audio --data $DATA --cache $CACHE --device cuda 2>&1 | grep --line-buffered -E 'saved|embedded'

In [ ]:
!ls -la $CACHE   # expect 8 files: vis_ff, vis_wdf, vis_wdf_test, vis_celeb, vis_celeb_test, aud_ASVspoof2019_LA, aud_ASVspoof5, aud_InTheWild

## 7. Train the classifiers, calibrate, set thresholds (2–5 min)
Look at the printed JSON: `auc` should be ≥ 0.85 (face, in-domain test) and ≥ 0.90 (voice); `genuine_flagged` should be ≤ 0.03.

In [ ]:
!python scripts/train_heads.py fit --cache $CACHE 2>&1 | tee fit_log.txt | grep -v 'val auc'

## 8. End-to-end test: 383 held-out files through the real app (about 15–30 min)

In [ ]:
!python scripts/build_test_sets.py $DATA $E2E | tail -2
!mkdir -p e2e_results
!python scripts/evaluate_folder.py --device cuda --real $E2E/A_ff/real  --fake $E2E/A_ff/fake  --json e2e_results/A_ff.json  | tail -3
!python scripts/evaluate_folder.py --device cuda --real $E2E/B_wdf/real --fake $E2E/B_wdf/fake --json e2e_results/B_wdf.json | tail -3
!python scripts/evaluate_folder.py --device cuda --real $E2E/C_itw/real --fake $E2E/C_itw/fake --json e2e_results/C_itw.json | tail -3
!python scripts/evaluate_folder.py --device cuda --real $E2E/D_av/real                         --json e2e_results/D_av.json  | tail -3

## 9. Graphs and README results

In [ ]:
!python scripts/make_report.py --cache $CACHE --e2e e2e_results
from IPython.display import SVG, display
for f in ['data', 'layers', 'roc', 'scores', 'e2e']:
    display(SVG(f'docs/figures/{f}.svg'))

## 10. Wiring test with the real models (optional, 2 min)

In [ ]:
!pip install -q pytest scikit-image httpx && python -m pytest -q -m real_models

## 11. Send the results back

**Option A: push from here** (needs a GitHub personal access token with `repo` scope: GitHub → Settings → Developer settings → Tokens. Store it as Colab secret `GH_TOKEN`).

In [ ]:
from google.colab import userdata
tok = userdata.get('GH_TOKEN')
!git config user.name "Gutsy" && git config user.email "gutsy@users.noreply.github.com"
!git add src/avdd/assets/*_head.npz docs/figures README.md && git add -f fit_log.txt e2e_results/*.json
!git commit -m "Trained heads and measured results (Colab T4)"
!git push https://{tok}@github.com/Gutsyt/Multimodel_video_audio_deepfake.git HEAD:v2-update

**Option B: download a zip** and upload the files on GitHub (branch `v2-update` → *Add file → Upload files*, keeping the same folders).

In [ ]:
!zip -qr /content/avdd_results.zip src/avdd/assets/visual_head.npz src/avdd/assets/audio_head.npz docs/figures README.md fit_log.txt e2e_results
from google.colab import files
files.download('/content/avdd_results.zip')